## Libraries

In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import math

from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL

from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors


## Config

In [2]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")
TEST_START_DATE  = pd.Timestamp("2022-04-01")   # all dates >= this are test

# ---- Best hyperparameters from tuning (fill these in) ----
best_lag_set = [1, 2, 3, 12]  # example

best_params = {
    "gamma_space": 0.1,
    "n_space": 500,
    "gamma_time": 0.05,
    "n_time": 500,
    "gamma_socio": 0.1,
    "n_socio": 0,     # 0 => linear/raw socio block
    "w_space": 1.0,
    "w_time": 1.0,
    "w_socio": 1.0,
    "alpha": 0.1,      # ridge regularisation in feature space
}

# Rolling STL configuration (recommend matching CV design)
STL_PERIOD = 12
STL_MIN_HISTORY = 24
STL_WINDOW = 120  # None for expanding, or 120 for 10y rolling history

# --- feature lists (must match your data) ---
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4",
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

spatial_cols = ["centroid_x", "centroid_y", "CoL_distance_km"]
temporal_macro_cols = ["base_rate", "GDP", "CPIH", "sdlt_perc_threshold"]


## Metric functions

In [8]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return float(same_dir.mean()) if mask.any() else np.nan

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    if not mask.any():
        return np.nan
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]
    return float(np.mean(np.abs(y_gr - yhat_gr)))

def morans_i(
    residuals,
    xs,
    ys,
    k=5,
    eps=1e-8,
    symmetric=True,
    row_standardize=True,
    permutations=0,
    random_state=None,
):
    """
    Compute Moran's I for residuals using k-nearest neighbours
    with inverse-distance weights.
    """
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    N = len(residuals)
    if not (len(xs) == len(ys) == N):
        raise ValueError("residuals, xs, ys must all have the same length")

    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=min(k + 1, N)).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        # skip self if present in first position
        neigh = indices[i]
        dist = distances[i]
        if len(neigh) == 0:
            continue
        if neigh[0] == i:
            neigh_idx = neigh[1:]
            dist_idx = dist[1:]
        else:
            neigh_idx = neigh[:k]
            dist_idx = dist[:k]

        if len(neigh_idx) == 0:
            continue
        w = 1.0 / (dist_idx + eps)
        W[i, neigh_idx] = w

    if symmetric:
        W = 0.5 * (W + W.T)

    if row_standardize:
        row_sums = W.sum(axis=1, keepdims=True)
        W = np.where(row_sums > 0, W / (row_sums + eps), 0.0)

    S0 = W.sum()
    if S0 <= 0:
        return {"I": np.nan, "S0": float(S0), "permutations": None, "z_score": None, "p_value": None}

    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps
    I_obs = (N / S0) * (num / den)

    result = {"I": float(I_obs), "S0": float(S0), "permutations": None, "z_score": None, "p_value": None}

    if permutations > 0:
        rng = np.random.default_rng(random_state)
        perm_I = np.empty(permutations, dtype=float)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)

        mean_perm = perm_I.mean()
        std_perm = perm_I.std(ddof=1) + eps
        z = (I_obs - mean_perm) / std_perm

        extreme = np.sum(np.abs(perm_I - mean_perm) >= np.abs(I_obs - mean_perm))
        p_val = (extreme + 1) / (permutations + 1)

        result.update({"permutations": perm_I, "z_score": float(z), "p_value": float(p_val)})

    return result

# --- Interval / probabilistic metrics ---
def _phi(z):
    z = np.asarray(z, dtype=float)
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z * z)

def _Phi(z):
    # Normal CDF via erf (no scipy dependency)
    z = np.asarray(z, dtype=float)
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))

def crps_gaussian(y, mu, sigma, eps=1e-12):
    """
    CRPS for N(mu, sigma^2). Lower is better.
    Formula: CRPS = sigma * [ z*(2Φ(z)-1) + 2φ(z) - 1/sqrt(pi) ], z=(y-mu)/sigma
    """
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    sigma = np.maximum(sigma, eps)
    z = (y - mu) / sigma
    return float(np.mean(sigma * (z * (2.0 * _Phi(z) - 1.0) + 2.0 * _phi(z) - 1.0 / np.sqrt(np.pi))))

def picp(y, lo, hi):
    y = np.asarray(y, dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    return float(np.mean((y >= lo) & (y <= hi)))

def piw(lo, hi):
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    return float(np.mean(hi - lo))



## Load data

In [4]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.query('Date < "2024-03-31"').copy()
df = df.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)


## Rolling STL feature builder

In [5]:
def add_rolling_stl_components(
    df,
    entity_col,
    time_col,
    target_col,
    period=12,
    min_history=24,
    window=120,        # set None for expanding, or e.g. 120 to match your 10y window
    robust=True,
    show_progress=True,
):
    """
    For each LA, compute STL components at time t using only y up to time t.
    We assign the *last* STL values from the fitted history to that time t.

    IMPORTANT:
    - This creates stl_trend/stl_seasonal/stl_resid for each row.
    - You should only use *lags* of these components (e.g., lag1/lag12/lag24)
      when predicting y_t, otherwise you'd leak y_t into its own features.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    # Precompute total iterations for tqdm
    groups = list(df.groupby(entity_col))
    total_steps = sum(len(sub) for _, sub in groups)

    iterator = tqdm(
        groups,
        desc="Rolling STL per LA",
        total=len(groups),
        leave=True,
        disable=not show_progress,
    )

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        n = len(sub)

        for t in range(n):
            start = 0 if window is None else max(0, t - window + 1)
            hist = y[start : t + 1]

            if len(hist) < min_history or np.isnan(hist).any():
                continue

            try:
                res = STL(hist, period=period, robust=robust).fit()

                idx = sub.index[t]
                df.loc[idx, "stl_trend"]    = res.trend[-1]
                df.loc[idx, "stl_seasonal"] = res.seasonal[-1]
                df.loc[idx, "stl_resid"]    = res.resid[-1]

            except Exception:
                continue

    return df

## Training

In [6]:
df = add_rolling_stl_components(
    df,
    entity_col=ENTITY_COL,
    time_col=TIME_COL,
    target_col=TARGET_COL,
    period=STL_PERIOD,
    min_history=STL_MIN_HISTORY,
    window=STL_WINDOW,
    robust=True,
    show_progress=True,
)

# Global time features (safe: do not use TARGET_COL)
origin = df[TIME_COL].min()
df["t_month"] = (
    (df[TIME_COL].dt.year - origin.year) * 12
    + (df[TIME_COL].dt.month - origin.month)
).astype(float)

df["month_num"] = df[TIME_COL].dt.month.astype(float)
df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12.0)
df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12.0)

time_cols = ["t_month", "month_cos", "month_sin"]

# Lagged STL features (best_lag_set)
required_lag_cols = []
for lag in best_lag_set:
    for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
        col = f"{comp}_lag{lag}"
        df[col] = df.groupby(ENTITY_COL)[comp].shift(lag)
        required_lag_cols.append(col)

# Temporal cols used inside the time block = STL lags + selected macro series (if present)
temporal_cols = required_lag_cols + [c for c in temporal_macro_cols if c in df.columns]

# Socio-economic continuous = continuous minus (spatial + temporal_macro)
socio_cont_cols = [c for c in continuous_cols if c not in set(spatial_cols + temporal_macro_cols)]


## Train / Test split
mask_train = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
mask_test  = (df[TIME_COL] >= TEST_START_DATE)

df_train = df.loc[mask_train].copy()
df_test  = df.loc[mask_test].copy()

# Require all needed columns present
base_needed = continuous_cols + categorical_cols + time_cols + temporal_cols
df_train = df_train.dropna(subset=base_needed + [TARGET_COL])
df_test  = df_test.dropna(subset=base_needed + [TARGET_COL])

if df_train.empty or df_test.empty:
    raise ValueError("Train or test is empty after dropping NaNs. Check dates/features/lags/STL settings.")


## Build blocks (train/test)
X_space_tr = df_train[spatial_cols].copy()
X_space_te = df_test[spatial_cols].copy()

X_time_tr = df_train[time_cols + temporal_cols].copy()
X_time_te = df_test[time_cols + temporal_cols].copy()

X_socio_tr = df_train[socio_cont_cols + categorical_cols].copy()
X_socio_te = df_test[socio_cont_cols + categorical_cols].copy()

y_train = df_train[TARGET_COL].values.astype(float)
y_test  = df_test[TARGET_COL].values.astype(float)

# Scale blocks (fit on train only)
sc_space = StandardScaler()
X_space_tr.loc[:, spatial_cols] = sc_space.fit_transform(X_space_tr[spatial_cols].astype(float))
X_space_te.loc[:, spatial_cols] = sc_space.transform(X_space_te[spatial_cols].astype(float))

sc_time = StandardScaler()
time_block_cols = time_cols + temporal_cols
X_time_tr.loc[:, time_block_cols] = sc_time.fit_transform(X_time_tr[time_block_cols].astype(float))
X_time_te.loc[:, time_block_cols] = sc_time.transform(X_time_te[time_block_cols].astype(float))

sc_socio = StandardScaler()
X_socio_tr.loc[:, socio_cont_cols] = sc_socio.fit_transform(X_socio_tr[socio_cont_cols].astype(float))
X_socio_te.loc[:, socio_cont_cols] = sc_socio.transform(X_socio_te[socio_cont_cols].astype(float))

# Scale y (as in your tuning)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()
y_scale = float(y_scaler.scale_[0])  # std in original units


## Train final SparseGP_RFF (RFF blocks + ridge)
params = best_params

# RFF transforms
rff_space = RBFSampler(gamma=params["gamma_space"], n_components=params["n_space"], random_state=42)
Zs_tr = rff_space.fit_transform(X_space_tr)
Zs_te = rff_space.transform(X_space_te)

rff_time = RBFSampler(gamma=params["gamma_time"], n_components=params["n_time"], random_state=43)
Zt_tr = rff_time.fit_transform(X_time_tr)
Zt_te = rff_time.transform(X_time_te)

if int(params["n_socio"]) == 0:
    Ze_tr = X_socio_tr.values
    Ze_te = X_socio_te.values
else:
    rff_socio = RBFSampler(gamma=params["gamma_socio"], n_components=params["n_socio"], random_state=44)
    Ze_tr = rff_socio.fit_transform(X_socio_tr)
    Ze_te = rff_socio.transform(X_socio_te)

ws = np.sqrt(float(params["w_space"]))
wt = np.sqrt(float(params["w_time"]))
we = np.sqrt(float(params["w_socio"]))

Z_train = np.hstack([ws * Zs_tr, wt * Zt_tr, we * Ze_tr])
Z_test  = np.hstack([ws * Zs_te, wt * Zt_te, we * Ze_te])

alpha = float(params["alpha"])
model = Ridge(alpha=alpha, fit_intercept=True)
model.fit(Z_train, y_train_scaled)

# Point predictions
yhat_test_scaled = model.predict(Z_test)
yhat_test = y_scaler.inverse_transform(yhat_test_scaled.reshape(-1, 1)).ravel()

df_test = df_test.copy()
df_test["y_pred"] = yhat_test
df_test["resid"] = df_test[TARGET_COL] - df_test["y_pred"]


## Predictive uncertainty (Gaussian) from ridge-as-Bayes approximation
# Predictive variance: sigma^2 * (1 + z^T (Z^T Z + alpha I)^-1 z)
# where sigma^2 estimated from train residuals in scaled space.
yhat_train_scaled = model.predict(Z_train)
train_resid_scaled = y_train_scaled - yhat_train_scaled

n = Z_train.shape[0]
p = Z_train.shape[1]

# sigma_hat (scaled space) using residual MSE (with dof protection)
dof = max(n - p - 1, 1)
sigma2_hat_scaled = float((train_resid_scaled @ train_resid_scaled) / dof)
sigma_hat_train = float(np.sqrt(sigma2_hat_scaled) * y_scale)  # in original units (requested)

# Compute A^{-1} robustly
# Note: if alpha==0 and Z^T Z is singular, use pseudo-inverse
A = Z_train.T @ Z_train
if alpha > 0:
    A = A + alpha * np.eye(p)

try:
    # Prefer solve against identity (more stable than explicit inverse)
    A_inv = np.linalg.solve(A, np.eye(p))
except np.linalg.LinAlgError:
    A_inv = np.linalg.pinv(A)

# Predictive variance for test points
# var_scaled = sigma2_hat_scaled * (1 + diag(Z_test A_inv Z_test^T))
# Efficient diagonal computation:
ZA = Z_test @ A_inv
diag_quad = np.sum(ZA * Z_test, axis=1)  # z^T A_inv z
pred_var_scaled = sigma2_hat_scaled * (1.0 + diag_quad)
pred_sd = np.sqrt(np.maximum(pred_var_scaled, 0.0)) * y_scale  # back to original units

z_975 = 1.959963984540054  # approx for 97.5% quantile
df_test["y_pred_sd"] = pred_sd
df_test["pi95_lo"] = df_test["y_pred"] - z_975 * df_test["y_pred_sd"]
df_test["pi95_hi"] = df_test["y_pred"] + z_975 * df_test["y_pred_sd"]




Rolling STL per LA: 100%|██████████| 294/294 [05:13<00:00,  1.07s/it]
C:\Users\slong\AppData\Local\Temp\ipykernel_30204\2837362584.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.16186804 -1.16186804 -1.16186804 ...  4.12448929  4.12448929
  4.12448929]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_time_tr.loc[:, time_block_cols] = sc_time.fit_transform(X_time_tr[time_block_cols].astype(float))
C:\Users\slong\AppData\Local\Temp\ipykernel_30204\2837362584.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.16186804 -1.16186804 -1.16186804 ...  4.12448929  4.12448929
  4.12448929]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_time_te.loc[:, time_block_cols] = sc_time.transform(X_time_te[time_block_cols].astype(float))


## Evaluation

In [11]:
# Global accuracy
global_mae   = mae(y_test, yhat_test)
global_rmse  = rmse(y_test, yhat_test)
global_smape = smape(y_test, yhat_test)
global_mase  = mase(y_test, yhat_test, y_train, m=12)

# Interval / probabilistic metrics
PICP_95 = picp(df_test[TARGET_COL].values, df_test["pi95_lo"].values, df_test["pi95_hi"].values)
PIW_95  = piw(df_test["pi95_lo"].values, df_test["pi95_hi"].values)
CRPS    = crps_gaussian(df_test[TARGET_COL].values, df_test["y_pred"].values, df_test["y_pred_sd"].values)

print("=== Global accuracy (SparseGP_RFF) ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:,.3f}%")
print(f"MASE  : {global_mase:,.3f}")

print("\n=== Uncertainty / interval metrics ===")
print(f"Sigma_hat_train (orig units): {sigma_hat_train:,.4f}")
print(f"PICP_95  : {PICP_95:,.3f}")
print(f"PIW_95   : {PIW_95:,.3f}")
print(f"CRPS     : {CRPS:,.4f}")

# Across-LA consistency
la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g[TARGET_COL].values, g["y_pred"].values))
median_mae = float(la_mae.median())
p75_mae    = float(la_mae.quantile(0.75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# Spatio-temporal diagnostics
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_test
    .dropna(subset=["centroid_x", "centroid_y"])
    .sort_values(TIME_COL)
    .groupby(ENTITY_COL)
    .tail(1)
    .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

common = la_resid_mean.index.intersection(centroids.index)
la_resid_mean = la_resid_mean.loc[common]
centroids = centroids.loc[common]

mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

print(f"\nLAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

if len(centroids_valid) <= 1:
    I_moran = {"I": np.nan, "z_score": np.nan, "p_value": np.nan}
else:
    k_eff = min(5, len(centroids_valid) - 1)
    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_eff,
        permutations=999,
        random_state=42,
    )
    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
    print(f"Moran's I z score: {I_moran['z_score']:.4f}")
    print(f"Moran's I p value: {I_moran['p_value']:.4f}")

monthly_resid = df_test.groupby(TIME_COL)["resid"].mean().sort_index()
lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])
print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# Optional: direction & growth
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")



=== Global accuracy (SparseGP_RFF) ===
MAE   : 32,537.829
RMSE  : 40,992.823
sMAPE : 12.874%
MASE  : 1.546

=== Uncertainty / interval metrics ===
Sigma_hat_train (orig units): 11,827.3529
PICP_95  : 0.478
PIW_95   : 51,427.322
CRPS     : 26,833.4154

=== Across-LA consistency ===
Median LA MAE       : 31,409.436
75th percentile MAE : 36,249.486

LAs used for Moran's I: 294 / 294

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): 0.2244
Moran's I z score: 6.3723
Moran's I p value: 0.0010
Ljung–Box Q(12): stat=34.300, p=0.0006

=== Direction & growth ===
Directional accuracy (MoM sign)   : 0.455
Growth-rate error MAE (12-month)  : 0.1183


C:\Users\slong\AppData\Local\Temp\ipykernel_30204\2031922255.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g[TARGET_COL].values, g["y_pred"].values))


## Output

In [12]:
output_path = "../../results/sparsegp_rff_final_test_results.xlsx"

summary_df = pd.DataFrame([{
    "model_type": "SparseGP_RFF_STL",
    "best_lag_set": str(best_lag_set),
    "best_params": str(best_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": float(I_moran["I"]) if isinstance(I_moran, dict) else np.nan,
    "Morans_I_z": float(I_moran["z_score"]) if isinstance(I_moran, dict) and I_moran.get("z_score") is not None else np.nan,
    "Morans_I_p": float(I_moran["p_value"]) if isinstance(I_moran, dict) and I_moran.get("p_value") is not None else np.nan,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae,
    # ---- requested additions ----
    "PICP_95": PICP_95,
    "PIW_95": PIW_95,
    "CRPS": CRPS,
    "Sigma_hat_train": sigma_hat_train,
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test[[ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", "y_pred_sd", "pi95_lo", "pi95_hi", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")


Results saved to: ../../results/sparsegp_rff_final_test_results.xlsx
